# SAE Probing: Re-implementation of *Are Sparse Autoencoders Useful?*

**Paper:** Kantamneni et al. (2025). [arXiv:2502.16681](https://arxiv.org/abs/2502.16681)

---

### What this notebook does

1. **Loads** a set of binary classification datasets (subsets of the paper's 113)
2. **Extracts** LLM hidden states from Gemma-2-2B (smaller stand-in for 9B)
3. **Encodes** those states through a Sparse Autoencoder (Gemma Scope, 16k width)
4. **Trains probes** — both baseline (LogReg, PCA, KNN, XGBoost, MLP) and SAE-based
5. **Evaluates** using the *Quiver of Arrows* framework across 4 difficult regimes:
   - Standard conditions
   - Data scarcity
   - Class imbalance
   - Label noise

### Key question
> Do SAE probes consistently outperform simple activation-based probes?

**Paper's answer:** No — baseline methods match or exceed SAE probes in every regime.

## Setup
Install packages (run once, then comment out).

In [ ]:
# Uncomment and run once to install all dependencies
# !pip install torch transformers sae-lens scikit-learn xgboost anthropic \
#              datasets numpy pandas matplotlib seaborn tqdm huggingface-hub \
#              accelerate ipywidgets

In [ ]:
import os
import json
import pickle
import warnings
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import xgboost as xgb

from datasets import load_dataset

# SAE loading via sae-lens (wraps Gemma Scope, Llama Scope, etc.)
from sae_lens import SAE

# Claude API for autointerp (Section 4 of paper)
import anthropic

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("All imports successful.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

In [ ]:
# ============================================================
#  Central configuration — change values here to experiment
# ============================================================

CONFIG = {
    # ── Model ────────────────────────────────────────────────
    # Gemma-2-2B is the smaller stand-in (paper uses 9B).
    # Requires accepting the license on HuggingFace first:
    # https://huggingface.co/google/gemma-2-2b
    "model_name":   "google/gemma-2-2b",
    "target_layer": 12,          # Middle of 26 layers; paper shows this works best
    "batch_size":   8,           # Reduce to 4 if you run out of GPU memory
    "max_seq_len":  256,         # Truncate prompts longer than this
    "device":       "cuda" if torch.cuda.is_available() else "cpu",

    # ── SAE (Gemma Scope) ────────────────────────────────────
    # sae-lens release name and specific SAE id.
    # Full list of available SAEs: https://github.com/jbloomAus/SAELens
    "sae_release":  "gemma-scope-2b-pt-res",
    "sae_id":       "layer_12/width_16k/average_l0_71",

    # ── SAE Probe ────────────────────────────────────────────
    # k = number of SAE latents to use in the probe (Equation 1 in paper).
    # Paper uses k=16 (interpretable) and k=128 (performance).
    "k_values":     [16, 128],

    # ── Hyperparameter search ────────────────────────────────
    "n_hp_values":  10,          # Number of hyperparameter values to try per method
    "random_seed":  42,

    # ── Experiment settings ──────────────────────────────────
    "n_train_standard": 1024,   # Training examples in standard conditions
    "n_test_min":       100,    # Skip datasets with fewer test examples

    # ── Paths ────────────────────────────────────────────────
    "cache_dir":    "./cache",
    "results_dir":  "./results",
}

# Seed everything for reproducibility
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
torch.manual_seed(CONFIG["random_seed"])

Path(CONFIG["cache_dir"]).mkdir(exist_ok=True)
Path(CONFIG["results_dir"]).mkdir(exist_ok=True)

print(f"Device: {CONFIG['device']}")
print(f"Model:  {CONFIG['model_name']}")
print(f"SAE:    {CONFIG['sae_release']} / {CONFIG['sae_id']}")

---
## 1. Datasets

The paper uses 113 binary classification datasets. We implement a representative subset
of 10 datasets, all publicly available on HuggingFace.

Each dataset is returned as:
```python
{
    "name": str,                     # matches paper's dataset ID
    "train": [(prompt, label), ...], # list of (str, int) pairs
    "test":  [(prompt, label), ...],
}
```
Labels are always binary: `0` or `1`.

In [ ]:
def load_glue_cola() -> dict:
    """87_glue_cola: Linguistic acceptability. Target=1 if grammatically correct."""
    ds = load_dataset("glue", "cola")
    # GLUE test labels are hidden — use 'validation' as our test set
    train = [(ex["sentence"], ex["label"]) for ex in ds["train"]]
    test  = [(ex["sentence"], ex["label"]) for ex in ds["validation"]]
    return {"name": "87_glue_cola", "train": train, "test": test}


def load_glue_sst2() -> dict:
    """92_glue_sst2: Movie review sentiment. Target=1 if positive."""
    ds = load_dataset("glue", "sst2")
    train = [(ex["sentence"], ex["label"]) for ex in ds["train"]]
    test  = [(ex["sentence"], ex["label"]) for ex in ds["validation"]]
    return {"name": "92_glue_sst2", "train": train, "test": test}


def load_glue_mrpc() -> dict:
    """89_glue_mrpc: Paraphrase detection. Target=1 if sentences are paraphrases.
    Prompt = both sentences concatenated with [SEP].
    """
    ds = load_dataset("glue", "mrpc")
    def fmt(ex): return ex["sentence1"] + " [SEP] " + ex["sentence2"]
    train = [(fmt(ex), ex["label"]) for ex in ds["train"]]
    test  = [(fmt(ex), ex["label"]) for ex in ds["validation"]]
    return {"name": "89_glue_mrpc", "train": train, "test": test}


def load_glue_qnli() -> dict:
    """90_glue_qnli: Question-answer NLI. Target=1 if answer entails question.
    Raw label: 0=entailment, 1=not_entailment — we flip so 1=entailment.
    """
    ds = load_dataset("glue", "qnli")
    def fmt(ex): return f"Q: {ex['question']} A: {ex['sentence']}"
    train = [(fmt(ex), 1 - ex["label"]) for ex in ds["train"]]
    test  = [(fmt(ex), 1 - ex["label"]) for ex in ds["validation"]]
    return {"name": "90_glue_qnli", "train": train, "test": test}


def load_glue_qqp() -> dict:
    """91_glue_qqp: Quora question pairs. Target=1 if questions are duplicates."""
    ds = load_dataset("glue", "qqp")
    def fmt(ex): return ex["question1"] + " [SEP] " + ex["question2"]
    train = [(fmt(ex), ex["label"]) for ex in ds["train"]]
    test  = [(fmt(ex), ex["label"]) for ex in ds["validation"]]
    return {"name": "91_glue_qqp", "train": train, "test": test}


def load_glue_mnli_entailment() -> dict:
    """136_glue_mnli_entailment: Multi-genre NLI. Target=1 if entailment (label=0)."""
    ds = load_dataset("glue", "mnli")
    def fmt(ex): return ex["premise"] + " [SEP] " + ex["hypothesis"]
    # label: 0=entailment, 1=neutral, 2=contradiction
    # Binary: entailment vs non-entailment
    def to_binary(label): return 1 if label == 0 else 0
    train = [(fmt(ex), to_binary(ex["label"])) for ex in ds["train"]]
    test  = [(fmt(ex), to_binary(ex["label"])) for ex in ds["validation_matched"]]
    return {"name": "136_glue_mnli_entailment", "train": train, "test": test}


def load_sciq() -> dict:
    """36_sciq_tf: Science true/false. Target=1 if answer is correct.
    We create positive examples (question + correct answer) and
    negative examples (question + distractor).
    """
    ds = load_dataset("allenai/sciq")
    def make_pairs(split):
        pairs = []
        for ex in split:
            q = ex["question"]
            pairs.append((f"Q: {q} A: {ex['correct_answer']}", 1))
            if ex["distractor1"]:  # add one negative per question
                pairs.append((f"Q: {q} A: {ex['distractor1']}", 0))
        return pairs
    return {"name": "36_sciq_tf", "train": make_pairs(ds["train"]),
            "test": make_pairs(ds["test"])}


def load_piqa() -> dict:
    """44_phys_tf: Physical intuition Q&A. Target=1 if solution is correct.
    PIQA has a goal and two solutions — one correct, one incorrect.
    We create one positive and one negative example per item.
    """
    ds = load_dataset("piqa")
    def make_pairs(split):
        pairs = []
        for ex in split:
            g = ex["goal"]
            label = ex["label"]  # 0=sol1 correct, 1=sol2 correct
            pairs.append((f"Goal: {g} Solution: {ex['sol1']}", 1 - label))
            pairs.append((f"Goal: {g} Solution: {ex['sol2']}", label))
        return pairs
    return {"name": "44_phys_tf", "train": make_pairs(ds["train"]),
            "test": make_pairs(ds["validation"])}


def load_rotten_tomatoes() -> dict:
    """113_movie_sent: Movie review sentiment. Target=1 if positive review."""
    ds = load_dataset("rotten_tomatoes")
    train = [(ex["text"], ex["label"]) for ex in ds["train"]]
    test  = [(ex["text"], ex["label"]) for ex in ds["test"]]
    return {"name": "113_movie_sent", "train": train, "test": test}


def load_amazon_polarity() -> dict:
    """157_amazon_5star: Amazon product review sentiment.
    Target=1 if positive (5-star), 0 if negative (1-star).
    We use a small subset (5000 train, 2000 test) for speed.
    """
    ds = load_dataset("amazon_polarity")
    # Truncate reviews to 200 chars so prompts stay short
    train = [(ex["content"][:200], ex["label"]) for ex in ds["train"].select(range(5000))]
    test  = [(ex["content"][:200], ex["label"]) for ex in ds["test"].select(range(2000))]
    return {"name": "157_amazon_5star", "train": train, "test": test}


print("Dataset loader functions defined.")

In [ ]:
# ── Load all datasets ──────────────────────────────────────────────────────
# Each loader downloads the data from HuggingFace on first run
# and caches it locally in ~/.cache/huggingface/datasets/

DATASET_LOADERS = {
    "87_glue_cola":             load_glue_cola,
    "92_glue_sst2":             load_glue_sst2,
    "89_glue_mrpc":             load_glue_mrpc,
    "90_glue_qnli":             load_glue_qnli,
    "91_glue_qqp":              load_glue_qqp,
    "136_glue_mnli_entailment": load_glue_mnli_entailment,
    "36_sciq_tf":               load_sciq,
    "44_phys_tf":               load_piqa,
    "113_movie_sent":           load_rotten_tomatoes,
    "157_amazon_5star":         load_amazon_polarity,
}

print(f"Loading {len(DATASET_LOADERS)} datasets...")
DATASETS = {}
for name, loader_fn in tqdm(DATASET_LOADERS.items()):
    DATASETS[name] = loader_fn()

# ── Summary table ──────────────────────────────────────────────────────────
rows = []
for name, ds in DATASETS.items():
    y_train = [y for _, y in ds["train"]]
    y_test  = [y for _, y in ds["test"]]
    rows.append({
        "Dataset":     name,
        "Train size":  len(ds["train"]),
        "Test size":   len(ds["test"]),
        "Train pos%": f"{100 * sum(y_train) / len(y_train):.1f}%",
        "Test pos%":  f"{100 * sum(y_test)  / len(y_test):.1f}%",
    })

df_summary = pd.DataFrame(rows).set_index("Dataset")
print("\nDataset summary:")
display(df_summary)

---
## 2. Activation Extraction

We run every prompt through Gemma-2-2B and record the **last-token hidden state**
at layer 12. The last token carries information about the entire context.

Activations are cached to disk so we only run the model once.

In [ ]:
def load_model_and_tokenizer(config: dict):
    """Load the language model and tokenizer.

    Note: Gemma-2 requires accepting the HuggingFace license.
    Visit https://huggingface.co/google/gemma-2-2b and click 'Agree and access'.
    Then run: huggingface-cli login
    """
    print(f"Loading tokenizer from {config['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
    tokenizer.pad_token = tokenizer.eos_token  # Gemma doesn't have a pad token by default
    tokenizer.padding_side = "right"           # Pad on the right so last token is meaningful

    print(f"Loading model (this may take a few minutes)...")
    model = AutoModelForCausalLM.from_pretrained(
        config["model_name"],
        torch_dtype=torch.float32,   # Use float32 for SAE compatibility
        device_map="auto",           # Automatically distributes across GPUs / CPU
    )
    model.eval()  # Disable dropout
    print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
    return model, tokenizer


# Load model (or skip if already loaded in memory)
if 'model' not in dir() or model is None:
    model, tokenizer = load_model_and_tokenizer(CONFIG)
else:
    print("Model already loaded.")

In [ ]:
def extract_activations(
    prompts: List[str],
    model,
    tokenizer,
    layer_idx: int,
    batch_size: int,
    max_seq_len: int,
    device: str,
) -> np.ndarray:
    """Extract last-token hidden states at `layer_idx` for each prompt.

    Returns:
        activations: np.ndarray of shape (n_prompts, d_model)
    """
    model.eval()
    all_activations = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]

        # Tokenize with padding so all sequences in the batch have the same length
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_len,
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # hidden_states is a tuple of (n_layers + 1) tensors, each (batch, seq, d_model)
        # Index 0 = embedding layer, index L = after layer L
        hidden = outputs.hidden_states[layer_idx]  # (batch, seq_len, d_model)

        # Find the last *real* token position for each sequence
        # attention_mask = 1 for real tokens, 0 for padding
        last_token_idx = inputs["attention_mask"].sum(dim=1) - 1  # (batch,)
        batch_acts = hidden[torch.arange(len(batch)), last_token_idx]  # (batch, d_model)

        all_activations.append(batch_acts.cpu().float().numpy())

    return np.concatenate(all_activations, axis=0)  # (n_prompts, d_model)


def get_or_compute_activations(
    dataset_name: str,
    prompts: List[str],
    split: str,
    model,
    tokenizer,
    config: dict,
) -> np.ndarray:
    """Load cached activations if they exist, otherwise compute and cache them."""
    cache_path = Path(config["cache_dir"]) / f"{dataset_name}_{split}_acts.npy"

    if cache_path.exists():
        return np.load(cache_path)

    print(f"  Computing activations for {dataset_name}/{split} ({len(prompts)} prompts)...")
    acts = extract_activations(
        prompts,
        model, tokenizer,
        layer_idx=config["target_layer"],
        batch_size=config["batch_size"],
        max_seq_len=config["max_seq_len"],
        device=config["device"],
    )
    np.save(cache_path, acts)
    return acts


print("Activation extraction functions defined.")

In [ ]:
# ── Extract (or load) activations for every dataset ──────────────────────
# First run: slow (runs the language model).  Subsequent runs: fast (loads cache).

ACTIVATIONS = {}  # {dataset_name: {"train": np.ndarray, "test": np.ndarray}}

for name, ds in tqdm(DATASETS.items(), desc="Datasets"):
    train_prompts = [p for p, _ in ds["train"]]
    test_prompts  = [p for p, _ in ds["test"]]

    ACTIVATIONS[name] = {
        "train": get_or_compute_activations(name, train_prompts, "train", model, tokenizer, CONFIG),
        "test":  get_or_compute_activations(name, test_prompts,  "test",  model, tokenizer, CONFIG),
    }

# Sanity check: print shapes for one dataset
example_name = next(iter(ACTIVATIONS))
print(f"\nExample — {example_name}:")
print(f"  Train activations: {ACTIVATIONS[example_name]['train'].shape}")
print(f"  Test  activations: {ACTIVATIONS[example_name]['test'].shape}")

---
## 3. SAE Loading & Encoding

We load a **Sparse Autoencoder** from Gemma Scope (trained by Google DeepMind).
The SAE maps a dense activation vector of size `d_model=2304` to a sparse vector
of size `W=16384` (width=16k), where only ~71 entries are nonzero on average (L0=71).

Each nonzero index corresponds to a learned, potentially human-interpretable *feature*.

In [ ]:
def load_sae(config: dict):
    """Load the SAE from sae-lens (downloads from HuggingFace on first run)."""
    print(f"Loading SAE: {config['sae_release']} / {config['sae_id']}")
    sae, cfg_dict, log_sparsity = SAE.from_pretrained(
        release=config["sae_release"],
        sae_id=config["sae_id"],
        device=config["device"],
    )
    sae.eval()
    print(f"SAE loaded.  Width={cfg_dict.get('d_sae', '?')}, "
          f"d_in={cfg_dict.get('d_in', '?')}")
    return sae


def encode_with_sae(activations: np.ndarray, sae, batch_size: int = 512) -> np.ndarray:
    """Pass activation vectors through the SAE encoder.

    Args:
        activations: shape (n_prompts, d_model)
        sae:         loaded SAE object from sae-lens

    Returns:
        latents: shape (n_prompts, sae_width) — sparse feature activations
    """
    sae.eval()
    all_latents = []
    act_tensor = torch.tensor(activations, dtype=torch.float32)

    for i in range(0, len(activations), batch_size):
        batch = act_tensor[i : i + batch_size].to(next(sae.parameters()).device)
        with torch.no_grad():
            out = sae.encode(batch)
        # sae-lens returns either a Tensor or a named-tuple / dict with 'feature_acts'
        if isinstance(out, torch.Tensor):
            latents = out
        elif hasattr(out, "feature_acts"):
            latents = out.feature_acts
        else:
            latents = out["feature_acts"]
        all_latents.append(latents.cpu().float().numpy())

    return np.concatenate(all_latents, axis=0)  # (n_prompts, sae_width)


def get_or_compute_sae_latents(
    dataset_name: str,
    activations: np.ndarray,
    split: str,
    sae,
    config: dict,
) -> np.ndarray:
    """Load cached SAE latents or compute and cache them."""
    cache_path = Path(config["cache_dir"]) / f"{dataset_name}_{split}_sae.npy"

    if cache_path.exists():
        return np.load(cache_path)

    print(f"  Encoding {dataset_name}/{split} with SAE...")
    latents = encode_with_sae(activations, sae)
    np.save(cache_path, latents)
    return latents


# Load the SAE
sae = load_sae(CONFIG)

In [ ]:
# ── Encode activations with SAE for every dataset ─────────────────────────

SAE_LATENTS = {}  # {dataset_name: {"train": np.ndarray, "test": np.ndarray}}

for name in tqdm(ACTIVATIONS, desc="SAE encoding"):
    SAE_LATENTS[name] = {
        "train": get_or_compute_sae_latents(name, ACTIVATIONS[name]["train"], "train", sae, CONFIG),
        "test":  get_or_compute_sae_latents(name, ACTIVATIONS[name]["test"],  "test",  sae, CONFIG),
    }

# Sanity check
example_name = next(iter(SAE_LATENTS))
train_latents = SAE_LATENTS[example_name]["train"]
print(f"\nExample — {example_name}:")
print(f"  SAE latent shape: {train_latents.shape}")
print(f"  Average L0 (nonzero per vector): {(train_latents != 0).sum(1).mean():.1f}")
print(f"  Sparsity: {(train_latents == 0).mean():.3f}")

---
## 4. Probing Methods

We implement 6 probe types:

| Method | Runs on | Key hyperparameter |
|--------|---------|--------------------|
| Logistic Regression | activations | regularization C |
| PCA + LogReg | activations | # PCA components |
| K-Nearest Neighbors | activations | k neighbors |
| XGBoost | activations | multiple tree params |
| MLP | activations | depth, width, lr |
| **SAE Probe** | **SAE latents** | **k latents, C** |

All methods return `{"val_auc": float, "test_auc": float}` so they can be
compared uniformly in the Quiver of Arrows framework.

In [ ]:
def make_train_val_test_split(
    prompts_and_labels: List[Tuple[str, int]],
    activations: np.ndarray,
    n_train: Optional[int] = None,
    class_ratio: Optional[float] = None,
    label_noise_frac: float = 0.0,
    rng: np.random.Generator = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Split activations into train / val / test following the paper's strategy.

    The paper uses held-out test sets from the original datasets.  Here we
    reserve the last 20% of the training data as validation and use all test
    data (stored separately in DATASETS) as the test set.

    Args:
        prompts_and_labels: list of (prompt, label) for the TRAIN split
        activations:        corresponding activation matrix (n_train_full, d_model)
        n_train:            number of training examples to use (data scarcity regime)
        class_ratio:        fraction of positive class (class imbalance regime)
        label_noise_frac:   fraction of labels to randomly flip (label noise regime)
        rng:                numpy random generator for reproducibility

    Returns:
        X_train, y_train, X_val, y_val  — for fitting + selecting hyperparameters
        (test set is passed separately since it always uses the full held-out data)
    """
    if rng is None:
        rng = np.random.default_rng(CONFIG["random_seed"])

    labels = np.array([y for _, y in prompts_and_labels])
    X_all  = activations
    y_all  = labels
    n_all  = len(y_all)

    # ── Class imbalance: subsample to desired ratio ────────────────────
    if class_ratio is not None:
        idx_pos = np.where(y_all == 1)[0]
        idx_neg = np.where(y_all == 0)[0]
        n_pos   = max(1, int(n_all * class_ratio))
        n_neg   = max(1, n_all - n_pos)
        idx_pos = rng.choice(idx_pos, size=min(n_pos, len(idx_pos)), replace=False)
        idx_neg = rng.choice(idx_neg, size=min(n_neg, len(idx_neg)), replace=False)
        idx     = rng.permutation(np.concatenate([idx_pos, idx_neg]))
        X_all, y_all = X_all[idx], y_all[idx]
        n_all = len(y_all)

    # ── Data scarcity: limit number of training examples ───────────────
    if n_train is not None and n_train < n_all:
        idx = rng.choice(n_all, size=n_train, replace=False)
        X_all, y_all = X_all[idx], y_all[idx]
        n_all = n_train

    # ── 80 / 20 train / validation split (paper uses this for n > 128) ─
    split_pt = max(1, int(0.8 * n_all))
    X_train, y_train = X_all[:split_pt], y_all[:split_pt]
    X_val,   y_val   = X_all[split_pt:], y_all[split_pt:]

    # ── Label noise: randomly flip a fraction of training labels ───────
    if label_noise_frac > 0:
        n_flip  = max(0, int(label_noise_frac * len(y_train)))
        flip_idx = rng.choice(len(y_train), size=n_flip, replace=False)
        y_train  = y_train.copy()
        y_train[flip_idx] = 1 - y_train[flip_idx]  # flip 0→1, 1→0

    return X_train, y_train, X_val, y_val


def safe_auc(y_true, y_score) -> float:
    """Compute AUC, returning 0.5 if only one class is present."""
    if len(np.unique(y_true)) < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_score))


print("Splitting utilities defined.")

In [ ]:
# ── Hyperparameter grids (Table C.3 in paper) ─────────────────────────────

def make_hp_grids(n_train: int, n_features: int, rng: np.random.Generator) -> dict:
    """Build hyperparameter grids for all methods based on dataset size.

    We follow the paper's hyperparameter ranges as closely as possible.
    """
    N = CONFIG["n_hp_values"]

    grids = {}

    # Logistic Regression — regularization strength C (inverse of lambda)
    grids["logreg"] = np.logspace(-5, 5, N)          # C in [1e-5, 1e5]

    # PCA + LogReg — number of components to keep
    max_pca = min(n_train - 1, n_features, 100)
    grids["pca"] = np.unique(np.logspace(0, np.log10(max(2, max_pca)), N).astype(int))

    # KNN — number of neighbours
    max_k = min(100, n_train - 1)
    grids["knn"] = np.unique(np.logspace(0, np.log10(max(2, max_k)), N).astype(int))

    # XGBoost — random search over multiple hyperparameters
    grids["xgb"] = [{
        "n_estimators":     rng.choice([50, 100, 150, 200, 250]),
        "max_depth":        rng.integers(2, 6),
        "learning_rate":    10 ** rng.uniform(-3, -1),
        "subsample":        rng.uniform(0.7, 1.0),
        "colsample_bytree": rng.uniform(0.7, 1.0),
        "reg_alpha":        10 ** rng.uniform(-3, 1),
        "reg_lambda":       10 ** rng.uniform(-3, 1),
        "min_child_weight": rng.integers(1, 10),
    } for _ in range(N)]

    # MLP — depth, hidden layer size, learning rate, weight decay
    hidden_options = [16, 32, 64]
    grids["mlp"] = [{
        "hidden_layer_sizes": tuple([rng.choice(hidden_options)] * rng.integers(1, 4)),
        "learning_rate_init": 10 ** rng.uniform(-4, -2),
        "alpha":              10 ** rng.uniform(-5, -2),   # L2 weight decay
        "activation":         "relu",
        "solver":             "adam",
    } for _ in range(N)]

    # SAE probe — regularization C (uses L1, so different grid from baseline LogReg)
    grids["sae"] = np.logspace(-5, 5, N)

    return grids


print("Hyperparameter grid builder defined.")

In [ ]:
# ── Baseline probing functions ─────────────────────────────────────────────
#
# All functions have the same signature:
#   (X_train, y_train, X_val, y_val, X_test, y_test, hp_grid)
#   → {"val_auc": float, "test_auc": float}
#
# We search over hp_grid, select the best setting by validation AUC,
# and report its test AUC.

def probe_logreg(X_tr, y_tr, X_va, y_va, X_te, y_te, C_values) -> dict:
    """L2-regularised logistic regression."""
    best_val, best_test = -1, 0.5
    for C in C_values:
        try:
            clf = LogisticRegression(C=C, penalty="l2", max_iter=500,
                                     random_state=CONFIG["random_seed"])
            clf.fit(X_tr, y_tr)
            v = safe_auc(y_va, clf.predict_proba(X_va)[:, 1])
            t = safe_auc(y_te, clf.predict_proba(X_te)[:, 1])
            if v > best_val:
                best_val, best_test = v, t
        except Exception:
            continue
    return {"val_auc": best_val, "test_auc": best_test}


def probe_pca_logreg(X_tr, y_tr, X_va, y_va, X_te, y_te, n_comp_values) -> dict:
    """PCA dimensionality reduction followed by logistic regression."""
    best_val, best_test = -1, 0.5
    for n in n_comp_values:
        try:
            n = min(n, X_tr.shape[0] - 1, X_tr.shape[1])
            pca = PCA(n_components=n)
            Xtr_r = pca.fit_transform(X_tr)
            Xva_r = pca.transform(X_va)
            Xte_r = pca.transform(X_te)
            clf = LogisticRegression(max_iter=500, random_state=CONFIG["random_seed"])
            clf.fit(Xtr_r, y_tr)
            v = safe_auc(y_va, clf.predict_proba(Xva_r)[:, 1])
            t = safe_auc(y_te, clf.predict_proba(Xte_r)[:, 1])
            if v > best_val:
                best_val, best_test = v, t
        except Exception:
            continue
    return {"val_auc": best_val, "test_auc": best_test}


def probe_knn(X_tr, y_tr, X_va, y_va, X_te, y_te, k_values) -> dict:
    """K-Nearest Neighbours with cosine distance."""
    best_val, best_test = -1, 0.5
    for k in k_values:
        try:
            clf = KNeighborsClassifier(n_neighbors=min(k, len(X_tr) - 1))
            clf.fit(X_tr, y_tr)
            v = safe_auc(y_va, clf.predict_proba(X_va)[:, 1])
            t = safe_auc(y_te, clf.predict_proba(X_te)[:, 1])
            if v > best_val:
                best_val, best_test = v, t
        except Exception:
            continue
    return {"val_auc": best_val, "test_auc": best_test}


def probe_xgboost(X_tr, y_tr, X_va, y_va, X_te, y_te, param_list) -> dict:
    """Gradient-boosted trees (XGBoost) with random hyperparameter search."""
    best_val, best_test = -1, 0.5
    for params in param_list:
        try:
            clf = xgb.XGBClassifier(**params, use_label_encoder=False,
                                    eval_metric="logloss", verbosity=0,
                                    random_state=CONFIG["random_seed"])
            clf.fit(X_tr, y_tr)
            v = safe_auc(y_va, clf.predict_proba(X_va)[:, 1])
            t = safe_auc(y_te, clf.predict_proba(X_te)[:, 1])
            if v > best_val:
                best_val, best_test = v, t
        except Exception:
            continue
    return {"val_auc": best_val, "test_auc": best_test}


def probe_mlp(X_tr, y_tr, X_va, y_va, X_te, y_te, param_list) -> dict:
    """Multi-layer perceptron; inputs are z-scored before training."""
    best_val, best_test = -1, 0.5
    scaler = StandardScaler().fit(X_tr)
    Xtr_s = scaler.transform(X_tr)
    Xva_s = scaler.transform(X_va)
    Xte_s = scaler.transform(X_te)
    for params in param_list:
        try:
            clf = MLPClassifier(**params, max_iter=200, random_state=CONFIG["random_seed"])
            clf.fit(Xtr_s, y_tr)
            v = safe_auc(y_va, clf.predict_proba(Xva_s)[:, 1])
            t = safe_auc(y_te, clf.predict_proba(Xte_s)[:, 1])
            if v > best_val:
                best_val, best_test = v, t
        except Exception:
            continue
    return {"val_auc": best_val, "test_auc": best_test}


print("Baseline probe functions defined.")

In [ ]:
# ── SAE Probe (Section 2.2 in paper) ─────────────────────────────────────

def select_top_k_latents(Z_train: np.ndarray, y_train: np.ndarray, k: int) -> np.ndarray:
    """Select the k SAE latents with the largest mean activation difference
    between the two classes (Equation 1 from the paper).

    Intuitively: we want features that 'fire' differently for class 0 vs class 1.

    Args:
        Z_train: SAE latents for training examples, shape (n_train, sae_width)
        y_train: binary labels, shape (n_train,)
        k:       number of latents to select

    Returns:
        indices: shape (k,) — indices into the SAE latent dimension
    """
    mean_1 = Z_train[y_train == 1].mean(axis=0)   # mean activation for positive class
    mean_0 = Z_train[y_train == 0].mean(axis=0)   # mean activation for negative class
    diff   = np.abs(mean_1 - mean_0)              # absolute difference per latent
    return np.argsort(diff)[-k:]                   # indices of top-k largest differences


def probe_sae(Z_tr, y_tr, Z_va, y_va, Z_te, y_te, k: int, C_values) -> dict:
    """SAE probe: top-k latent selection + L1-regularised logistic regression.

    Uses L1 regularisation (encouraging further sparsity in probe weights)
    rather than L2 used by the baseline logistic regression.

    Args:
        Z_*:      SAE latent matrices for train / val / test
        k:        number of top latents to use
        C_values: regularisation strength values to try

    Returns:
        {"val_auc": float, "test_auc": float,
         "top_k_indices": np.ndarray}  — indices useful for interpretability
    """
    # Step 1: select the k most discriminative latent features
    indices = select_top_k_latents(Z_tr, y_tr, k)

    # Step 2: slice to those k features only
    Ztr_k = Z_tr[:, indices]
    Zva_k = Z_va[:, indices]
    Zte_k = Z_te[:, indices]

    # Step 3: grid search over L1 regularisation
    best_val, best_test = -1, 0.5
    for C in C_values:
        try:
            clf = LogisticRegression(
                C=C, penalty="l1", solver="liblinear",
                max_iter=500, random_state=CONFIG["random_seed"]
            )
            clf.fit(Ztr_k, y_tr)
            v = safe_auc(y_va, clf.predict_proba(Zva_k)[:, 1])
            t = safe_auc(y_te, clf.predict_proba(Zte_k)[:, 1])
            if v > best_val:
                best_val, best_test = v, t
        except Exception:
            continue

    return {"val_auc": best_val, "test_auc": best_test, "top_k_indices": indices}


print("SAE probe functions defined.")

In [ ]:
def run_all_probes(
    dataset_name: str,
    X_train: np.ndarray, y_train: np.ndarray,
    X_val:   np.ndarray, y_val:   np.ndarray,
    X_test:  np.ndarray, y_test:  np.ndarray,
    Z_train: np.ndarray,  # SAE latents (train)
    Z_val:   np.ndarray,  # SAE latents (val)
    Z_test:  np.ndarray,  # SAE latents (test)
    k_values: List[int],
    rng: np.random.Generator,
) -> dict:
    """Run all 5 baseline probes + SAE probes on one dataset split.

    Returns a dict: {method_name: {"val_auc": float, "test_auc": float}}
    """
    n_train   = len(y_train)
    n_features = X_train.shape[1]
    grids = make_hp_grids(n_train, n_features, rng)

    results = {}

    # ── Baseline probes (on raw activations) ────────────────────────────
    results["logreg"] = probe_logreg(
        X_train, y_train, X_val, y_val, X_test, y_test, grids["logreg"])
    results["pca"]    = probe_pca_logreg(
        X_train, y_train, X_val, y_val, X_test, y_test, grids["pca"])
    results["knn"]    = probe_knn(
        X_train, y_train, X_val, y_val, X_test, y_test, grids["knn"])
    results["xgb"]    = probe_xgboost(
        X_train, y_train, X_val, y_val, X_test, y_test, grids["xgb"])
    results["mlp"]    = probe_mlp(
        X_train, y_train, X_val, y_val, X_test, y_test, grids["mlp"])

    # ── SAE probes (on sparse latents) for each k ───────────────────────
    for k in k_values:
        key = f"sae_k{k}"
        results[key] = probe_sae(
            Z_train, y_train, Z_val, y_val, Z_test, y_test,
            k=k, C_values=grids["sae"]
        )

    return results


print("run_all_probes() defined.")

---
## 5. Quiver of Arrows Evaluation

The **Quiver of Arrows** (Section 2.4 of paper) is the key evaluation framework.

**Idea:** A practitioner has a 'quiver' of methods. They run all of them, pick the best
by *validation* AUC, and deploy that method. We measure whether adding SAE probes to
the quiver increases the final *test* AUC.

```
Baseline quiver:  {logreg, pca, knn, xgb, mlp}
SAE quiver:       {logreg, pca, knn, xgb, mlp, sae_k16, sae_k128}

SAE gain = quiver_auc(SAE quiver) − quiver_auc(Baseline quiver)
```

A positive gain means SAE probes helped; negative means they hurt.

In [ ]:
BASELINE_METHODS = ["logreg", "pca", "knn", "xgb", "mlp"]


def quiver_of_arrows(results: dict, method_names: List[str]) -> Tuple[str, float]:
    """Select the method with highest validation AUC; return its test AUC.

    Args:
        results:      {method: {"val_auc": float, "test_auc": float}}
        method_names: which methods to include in this quiver

    Returns:
        (best_method_name, best_test_auc)
    """
    available = {m: results[m] for m in method_names if m in results}
    if not available:
        return "none", 0.5
    best = max(available, key=lambda m: available[m]["val_auc"])
    return best, available[best]["test_auc"]


def evaluate_dataset(
    dataset_name: str,
    results_by_split: dict,   # output of run_all_probes
    k_values: List[int],
) -> dict:
    """Compute quiver AUC with and without SAE probes and return the delta.

    Returns a summary dict for this dataset.
    """
    sae_method_names = [f"sae_k{k}" for k in k_values]

    # Best test AUC achievable using only baseline methods
    _, baseline_test_auc = quiver_of_arrows(results_by_split, BASELINE_METHODS)

    # Best test AUC when SAE probes are added to the quiver
    _, sae_test_auc = quiver_of_arrows(results_by_split, BASELINE_METHODS + sae_method_names)

    # Individual method test AUCs (for inspection)
    per_method = {m: results_by_split[m]["test_auc"]
                  for m in BASELINE_METHODS + sae_method_names
                  if m in results_by_split}

    return {
        "dataset":           dataset_name,
        "baseline_test_auc": baseline_test_auc,
        "sae_test_auc":      sae_test_auc,
        "sae_delta":         sae_test_auc - baseline_test_auc,   # >0 = SAE helped
        "per_method":        per_method,
    }


print("Quiver of Arrows framework defined.")

---
## 6. Experiments

We now run all four regimes. Each regime has a parameter that controls the difficulty:

| Regime | Parameter | Range |
|--------|-----------|-------|
| Standard | — | Full data, balanced |
| Data scarcity | n_train | 2 → 1024 (log-spaced) |
| Class imbalance | ratio (pos class %) | 5% → 95% |
| Label noise | frac (labels flipped) | 0% → 50% |

In [ ]:
# ── Experiment 1: Standard Conditions ─────────────────────────────────────
# Use up to 1024 training examples, balanced classes, no noise.

def run_standard_experiment(datasets, activations, sae_latents, config) -> pd.DataFrame:
    """Run the standard conditions experiment across all datasets.

    Returns a DataFrame with one row per dataset.
    """
    rng = np.random.default_rng(config["random_seed"])
    rows = []

    for name in tqdm(datasets, desc="Standard experiment"):
        ds = datasets[name]
        y_test = np.array([y for _, y in ds["test"]])

        # Skip datasets with too few test examples
        if len(y_test) < config["n_test_min"] or len(np.unique(y_test)) < 2:
            continue

        # Build train / val split (cap at n_train_standard)
        X_tr, y_tr, X_va, y_va = make_train_val_test_split(
            ds["train"],
            activations[name]["train"],
            n_train=config["n_train_standard"],
            rng=rng,
        )
        # Same split for SAE latents
        n_split = len(y_tr)  # number of training examples after split
        Z_all   = sae_latents[name]["train"]
        Z_tr    = Z_all[:n_split]
        Z_va    = Z_all[n_split : n_split + len(y_va)]
        Z_te    = sae_latents[name]["test"]

        X_te    = activations[name]["test"]

        # Run all probes
        probe_results = run_all_probes(
            name, X_tr, y_tr, X_va, y_va, X_te, y_test,
            Z_tr, Z_va, Z_te,
            k_values=config["k_values"], rng=rng,
        )

        # Evaluate with Quiver of Arrows
        summary = evaluate_dataset(name, probe_results, config["k_values"])
        rows.append(summary)

    return pd.DataFrame(rows).set_index("dataset")


print("Running standard conditions experiment...")
df_standard = run_standard_experiment(DATASETS, ACTIVATIONS, SAE_LATENTS, CONFIG)

print(f"\nCompleted {len(df_standard)} datasets.")
print(f"\nMean baseline test AUC: {df_standard['baseline_test_auc'].mean():.4f}")
print(f"Mean SAE test AUC:      {df_standard['sae_test_auc'].mean():.4f}")
print(f"Mean SAE delta:         {df_standard['sae_delta'].mean():.4f} "
      f"(+= SAE helped, -= baseline better)")
display(df_standard[["baseline_test_auc", "sae_test_auc", "sae_delta"]].round(4))

In [ ]:
# ── Experiment 2: Data Scarcity ────────────────────────────────────────────
# Vary the number of training examples from 2 to 1024 (log-spaced).
# Key question: do SAE probes help when labelled data is scarce?

def run_scarcity_experiment(datasets, activations, sae_latents, config) -> pd.DataFrame:
    """For each dataset, sweep n_train over 20 log-spaced values.

    Returns a long-format DataFrame with columns:
        dataset, n_train, baseline_test_auc, sae_test_auc, sae_delta
    """
    rng = np.random.default_rng(config["random_seed"])

    # 20 log-spaced values from 2 to 1024 (matching the paper)
    n_train_values = np.unique(np.logspace(np.log10(2), np.log10(1024), 20).astype(int))

    rows = []
    for name in tqdm(datasets, desc="Scarcity experiment"):
        ds = datasets[name]
        y_test = np.array([y for _, y in ds["test"]])
        if len(y_test) < config["n_test_min"] or len(np.unique(y_test)) < 2:
            continue

        X_te = activations[name]["test"]
        Z_te = sae_latents[name]["test"]

        for n_train in n_train_values:
            X_tr, y_tr, X_va, y_va = make_train_val_test_split(
                ds["train"], activations[name]["train"],
                n_train=int(n_train), rng=rng,
            )
            if len(np.unique(y_tr)) < 2:  # skip if only one class present
                continue

            n_split = len(y_tr)
            Z_all = sae_latents[name]["train"]
            Z_tr  = Z_all[:n_split]
            Z_va  = Z_all[n_split : n_split + len(y_va)]

            probe_results = run_all_probes(
                name, X_tr, y_tr, X_va, y_va, X_te, y_test,
                Z_tr, Z_va, Z_te, k_values=config["k_values"], rng=rng,
            )
            summary = evaluate_dataset(name, probe_results, config["k_values"])
            summary["n_train"] = n_train
            rows.append(summary)

    return pd.DataFrame(rows)


print("Running data scarcity experiment (this takes a while)...")
df_scarcity = run_scarcity_experiment(DATASETS, ACTIVATIONS, SAE_LATENTS, CONFIG)

# Average delta across datasets at each n_train level
scarcity_avg = df_scarcity.groupby("n_train")["sae_delta"].mean()
print(f"\nAverage SAE delta across n_train values:")
print(scarcity_avg.round(4).to_string())

In [ ]:
# ── Experiment 3: Class Imbalance ─────────────────────────────────────────
# Vary the fraction of positive-class examples in training.
# Tests whether SAE probes handle skewed class distributions better.

def run_imbalance_experiment(datasets, activations, sae_latents, config) -> pd.DataFrame:
    """Sweep class ratio (positive class fraction) over 19 values in [0.05, 0.95]."""
    rng = np.random.default_rng(config["random_seed"])

    # 19 linearly spaced ratio values (matching the paper)
    ratio_values = np.linspace(0.05, 0.95, 19)

    rows = []
    for name in tqdm(datasets, desc="Imbalance experiment"):
        ds = datasets[name]
        y_test = np.array([y for _, y in ds["test"]])
        if len(y_test) < config["n_test_min"] or len(np.unique(y_test)) < 2:
            continue

        X_te = activations[name]["test"]
        Z_te = sae_latents[name]["test"]

        for ratio in ratio_values:
            X_tr, y_tr, X_va, y_va = make_train_val_test_split(
                ds["train"], activations[name]["train"],
                class_ratio=float(ratio), rng=rng,
            )
            if len(np.unique(y_tr)) < 2:
                continue

            n_split = len(y_tr)
            Z_all = sae_latents[name]["train"]
            Z_tr  = Z_all[:n_split]
            Z_va  = Z_all[n_split : n_split + len(y_va)]

            probe_results = run_all_probes(
                name, X_tr, y_tr, X_va, y_va, X_te, y_test,
                Z_tr, Z_va, Z_te, k_values=config["k_values"], rng=rng,
            )
            summary = evaluate_dataset(name, probe_results, config["k_values"])
            summary["class_ratio"] = ratio
            rows.append(summary)

    return pd.DataFrame(rows)


print("Running class imbalance experiment...")
df_imbalance = run_imbalance_experiment(DATASETS, ACTIVATIONS, SAE_LATENTS, CONFIG)

imbalance_avg = df_imbalance.groupby("class_ratio")["sae_delta"].mean()
print(f"\nAverage SAE delta across class ratios:")
print(imbalance_avg.round(4).to_string())

In [ ]:
# ── Experiment 4: Label Noise ──────────────────────────────────────────────
# Randomly flip a fraction of training labels.
#
# Note: for this regime the validation set is also corrupted, so using
# validation AUC to select the method is unreliable. The paper therefore
# compares logistic regression vs. the SAE probe head-to-head (not quiver).

def run_noise_experiment(datasets, activations, sae_latents, config) -> pd.DataFrame:
    """Sweep label noise fraction over 11 values in [0, 0.5].

    Compares baseline logistic regression vs. SAE probe (k=128) directly,
    since the quiver approach is unreliable when validation labels are noisy.
    """
    rng = np.random.default_rng(config["random_seed"])

    # 11 linearly spaced noise fractions (matching the paper)
    noise_values = np.linspace(0.0, 0.5, 11)

    rows = []
    for name in tqdm(datasets, desc="Label noise experiment"):
        ds = datasets[name]
        y_test = np.array([y for _, y in ds["test"]])
        if len(y_test) < config["n_test_min"] or len(np.unique(y_test)) < 2:
            continue

        X_te = activations[name]["test"]
        Z_te = sae_latents[name]["test"]

        for frac in noise_values:
            X_tr, y_tr, X_va, y_va = make_train_val_test_split(
                ds["train"], activations[name]["train"],
                label_noise_frac=float(frac), rng=rng,
            )
            if len(np.unique(y_tr)) < 2:
                continue

            n_split = len(y_tr)
            Z_all = sae_latents[name]["train"]
            Z_tr  = Z_all[:n_split]
            Z_va  = Z_all[n_split : n_split + len(y_va)]

            # Logistic regression baseline
            C_values = np.logspace(-5, 5, config["n_hp_values"])
            lr_result = probe_logreg(X_tr, y_tr, X_va, y_va, X_te, y_test, C_values)

            # SAE probe (k=128, the performance setting)
            sae_result = probe_sae(
                Z_tr, y_tr, Z_va, y_va, Z_te, y_test,
                k=128, C_values=C_values,
            )

            rows.append({
                "dataset":       name,
                "noise_frac":    frac,
                "logreg_test":   lr_result["test_auc"],
                "sae_test":      sae_result["test_auc"],
                "sae_delta":     sae_result["test_auc"] - lr_result["test_auc"],
            })

    return pd.DataFrame(rows)


print("Running label noise experiment...")
df_noise = run_noise_experiment(DATASETS, ACTIVATIONS, SAE_LATENTS, CONFIG)

noise_avg = df_noise.groupby("noise_frac")["sae_delta"].mean()
print(f"\nAverage SAE delta across noise fractions:")
print(noise_avg.round(4).to_string())

---
## 7. Results & Visualisation

We reproduce the key figures from the paper:
- **Figure 1 style**: Bar chart comparing baseline vs SAE AUC across regimes
- **Figure 6 style**: SAE Δ-AUC as a function of each regime's parameter

In [ ]:
# ── Save all results to disk ───────────────────────────────────────────────
results_path = Path(CONFIG["results_dir"])

df_standard.to_csv(results_path / "standard.csv")
df_scarcity.to_csv(results_path / "scarcity.csv", index=False)
df_imbalance.to_csv(results_path / "imbalance.csv", index=False)
df_noise.to_csv(results_path / "noise.csv", index=False)

print("Results saved to", results_path)

In [ ]:
# ── Figure 1 reproduction: mean test AUC per regime ───────────────────────
# Each bar shows the average test AUC of the best method (baseline quiver
# or SAE quiver) across all datasets in that regime.

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)
fig.suptitle("SAE Probes vs Baseline Probes — Mean Test AUC by Regime",
             fontsize=13, fontweight="bold")

regime_data = [
    ("Standard",        df_standard[["baseline_test_auc", "sae_test_auc"]].mean()),
    ("Data Scarcity",   df_scarcity.groupby("dataset")[["baseline_test_auc", "sae_test_auc"]].mean().mean()),
    ("Class Imbalance", df_imbalance.groupby("dataset")[["baseline_test_auc", "sae_test_auc"]].mean().mean()),
    ("Label Noise",     df_noise.groupby("dataset")[["logreg_test", "sae_test"]].mean().mean()),
]

for ax, (label, means) in zip(axes, regime_data):
    vals = means.values
    bars = ax.bar(["Baseline", "SAE Probe"], vals,
                  color=["#c0392b", "#2980b9"], width=0.5, edgecolor="black")
    ax.set_title(label, fontsize=11)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel("Mean Test AUC")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(results_path / "fig1_regime_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig1_regime_comparison.png")

In [ ]:
# ── Figure 6 reproduction: SAE Δ-AUC vs regime parameter ─────────────────
# Shows whether the SAE advantage (or disadvantage) varies with difficulty.
# Shaded area = 95% confidence interval across datasets.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("SAE Δ-AUC (SAE − Baseline) across regimes — positive = SAE helps",
             fontsize=12)

def plot_delta_with_ci(ax, df, x_col, delta_col, x_label, log_x=False):
    """Plot mean SAE delta with 95% CI, grouping by x_col."""
    grouped = df.groupby(x_col)[delta_col]
    mean  = grouped.mean()
    sem   = grouped.sem()
    ci95  = 1.96 * sem   # 95% CI
    xs    = mean.index

    ax.axhline(0, color="black", linestyle="--", linewidth=1, label="No change")
    ax.fill_between(xs, mean - ci95, mean + ci95, alpha=0.3, color="gray")
    ax.plot(xs, mean, color="black", linewidth=2)
    ax.set_xlabel(x_label)
    ax.set_ylabel("SAE Δ-AUC")
    if log_x:
        ax.set_xscale("log")


# Panel 1: Data Scarcity
plot_delta_with_ci(axes[0], df_scarcity, "n_train", "sae_delta",
                   "Number of Training Examples", log_x=True)
axes[0].set_title("Data Scarcity")

# Panel 2: Class Imbalance
plot_delta_with_ci(axes[1], df_imbalance, "class_ratio", "sae_delta",
                   "Ratio of Positive Class")
axes[1].set_title("Class Imbalance")

# Panel 3: Label Noise
plot_delta_with_ci(axes[2], df_noise, "noise_frac", "sae_delta",
                   "Fraction of Labels Corrupted")
axes[2].set_title("Label Noise")

plt.tight_layout()
plt.savefig(results_path / "fig6_sae_delta.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig6_sae_delta.png")

In [ ]:
# ── Per-dataset scatter (Figure 4 style) ──────────────────────────────────
# Each dot is one dataset.  Points above the diagonal = SAE helped.

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(
    df_standard["baseline_test_auc"],
    df_standard["sae_test_auc"],
    alpha=0.7, s=60, edgecolors="black", linewidths=0.5,
)
# Diagonal = no change
lims = [0.5, 1.0]
ax.plot(lims, lims, "k--", linewidth=1, label="No change")

mean_delta = df_standard["sae_delta"].mean()
ax.set_title(f"Standard Conditions\nMean Δ: {mean_delta:+.4f}", fontsize=11)
ax.set_xlabel("Baseline Test AUC (without SAE)")
ax.set_ylabel("SAE Test AUC (with SAE)")
ax.set_xlim(0.5, 1.02)
ax.set_ylim(0.5, 1.02)

# Label each dataset
for name, row in df_standard.iterrows():
    ax.annotate(name.split("_")[0],  # just the numeric ID
                (row["baseline_test_auc"], row["sae_test_auc"]),
                fontsize=6, xytext=(4, 4), textcoords="offset points")

ax.legend()
plt.tight_layout()
plt.savefig(results_path / "fig4_scatter_standard.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Bonus: Latent Interpretability with Claude API

Section 4.2 of the paper generates natural-language descriptions of SAE latents
using automatic interpretation (**autointerp**).

We implement a simplified version: for a chosen dataset, we find the top SAE latent
and ask Claude to describe what it seems to detect, given a sample of text snippets
where that latent fires most strongly.

In [ ]:
def get_top_activating_examples(
    latent_idx: int,
    prompts: List[str],
    Z: np.ndarray,
    n: int = 10,
) -> List[Tuple[str, float]]:
    """Return the n prompts where latent `latent_idx` fires most strongly."""
    scores = Z[:, latent_idx]
    top_idx = np.argsort(scores)[::-1][:n]
    return [(prompts[i], float(scores[i])) for i in top_idx]


def autointerp_latent(
    latent_idx: int,
    top_examples: List[Tuple[str, float]],
    dataset_name: str,
    api_key: Optional[str] = None,
) -> str:
    """Ask Claude to describe what the SAE latent appears to detect.

    Args:
        latent_idx:   index of the latent in the SAE
        top_examples: list of (text, activation_value) pairs that activated it most
        dataset_name: the probing dataset this latent is associated with
        api_key:      Anthropic API key (reads from ANTHROPIC_API_KEY env var if None)

    Returns:
        A natural language description of the latent.
    """
    client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))

    # Format the examples into a readable list
    examples_text = "\n".join(
        f"  {i+1}. [{score:.2f}] {text[:120]}"
        for i, (text, score) in enumerate(top_examples)
    )

    prompt = f"""You are interpreting a feature (neuron) from a Sparse Autoencoder (SAE)
trained on a large language model. The feature is latent index {latent_idx}.

Below are the text snippets where this feature fires most strongly
(activation value shown in brackets). The texts come from the dataset: {dataset_name}.

TOP ACTIVATING EXAMPLES:
{examples_text}

Based on these examples, write a concise 1-2 sentence description of what linguistic
pattern or concept this feature appears to detect. Be specific about the pattern."""

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text


# ── Example: interpret the top latent for GLUE CoLA ──────────────────────
target_dataset = "87_glue_cola"

if target_dataset in DATASETS and target_dataset in SAE_LATENTS:
    train_prompts = [p for p, _ in DATASETS[target_dataset]["train"]]
    Z_train       = SAE_LATENTS[target_dataset]["train"]
    y_train       = np.array([y for _, y in DATASETS[target_dataset]["train"]])

    # Find the latent with biggest class difference (the top-1 SAE probe feature)
    top_indices = select_top_k_latents(Z_train, y_train, k=1)
    top_latent  = int(top_indices[0])
    print(f"Top SAE latent for {target_dataset}: index {top_latent}")

    # Get the text snippets where it fires most
    top_examples = get_top_activating_examples(top_latent, train_prompts, Z_train, n=10)

    print(f"\nTop activating examples for latent {top_latent}:")
    for text, score in top_examples[:5]:
        print(f"  [{score:.3f}] {text[:100]}")

    # Call Claude API to interpret the latent
    # Set your API key: export ANTHROPIC_API_KEY=sk-ant-...
    try:
        description = autointerp_latent(top_latent, top_examples, target_dataset)
        print(f"\nClaude's interpretation of latent {top_latent}:")
        print(f"  {description}")
    except Exception as e:
        print(f"\nAPI call failed (check ANTHROPIC_API_KEY): {e}")
else:
    print(f"Dataset {target_dataset} not found — run data loading cells first.")

---
## Summary

This notebook re-implements the core pipeline from *Are Sparse Autoencoders Useful?*

### What to look for in your results

| Metric | Paper's finding | Your result |
|--------|----------------|-------------|
| Standard `sae_delta` | −0.003 ± 0.002 | *(fill in)* |
| Data scarcity `sae_delta` | near 0 across all n_train | *(fill in)* |
| Class imbalance `sae_delta` | near 0 across all ratios | *(fill in)* |
| Label noise `sae_delta` | near 0 or negative | *(fill in)* |

### Differences from the paper
1. **Model**: Gemma-2-2B instead of Gemma-2-9B (smaller = less capable representations)
2. **Datasets**: 10 representative datasets instead of 113
3. **SAE**: Width 16k, L0≈71 (paper also tests 131k and 1M width)

### Extensions to try
- Swap to `google/gemma-2-9b` (if you have ~18GB VRAM) and compare
- Try SAE width `131k` or `1M` by changing `CONFIG['sae_id']`
- Add the covariate shift (OOD) experiment from Section 3.3
- Implement multi-token pooling (Section 5) and the attention-pooled baseline
- Add autointerp for all datasets and inspect spurious latents